# PSBD end to end

This notebook runs the whole method on one backdoored model: attach a placement, run several perturbed passes, score every image, choose a rate, set a threshold, and read the verdict. It runs both named placements, PSBD-TM (`before_attention_norm`, `token_mask`) and PSBD-RD (`post_residual`, `dropout`), so the histograms below are the ones `scripts/paper/fig_psu_histograms.py` draws from the cache, computed directly instead of read back from disk.

In [1]:
import os
import sys
from pathlib import Path

# Anchor at the repository root so every default path in the library resolves the
# same way it does from a script, whichever directory the notebook was opened from.
REPO_ROOT = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import logging

logging.getLogger("lightning.fabric.utilities.seed").setLevel(logging.WARNING)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

import scripts.paper._style  # noqa: F401  the figure style every table in the paper uses

from data.registry import DATASET_REGISTRY
from defences.decision import (
    HEADLINE_QUANTILE,
    PLACEMENT_MATCH_TARGET,
    PSBD_QUANTILES,
    PUBLISHED_PLACEMENT,
    RECOMMENDED_PLACEMENT,
    detection_report,
    select_rate_adaptively,
    select_rate_at_matched_shift,
    select_rate_by_oracle,
    threshold_at_quantile,
)
from defences.inference import build_baseline_cache, compute_dropout_pass_probs
from defences.operators import build_operator
from models.positions import DROPOUT_CONFIGS
from defences.scores import psu_from_cache, psu_ratio_from_cache, shift_ratio, shift_target_histogram
from data.splits import build_psbd_loaders_from_checkpoint
from models.backbones import detect_architecture, load_checkpoint
from models.positions import plug_dropout, unplug_dropout

CHECKPOINT = "checkpoints/vit_cifar100_badnet_a2o_0_01/attack_result.pt"
assert RECOMMENDED_PLACEMENT == "before_attention_norm_token_mask"
assert PUBLISHED_PLACEMENT == "post_residual"
PLACEMENTS = {
    "PSBD-TM": (("before_attention_norm",), "token_mask"),
    "PSBD-RD": (DROPOUT_CONFIGS[PUBLISHED_PLACEMENT], "dropout"),
}
FORWARD_PASSES = 3  # k = 3, the paper's own choice
RATES = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8)
MAX_SAMPLES = 800

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


## The three splits

A fixed permutation of the clean test set, seeded once, gives a 2000-image validation set for the threshold and the rate rule, and an analysis pool split into a clean and a triggered half. The triggered half is a subset of the clean half, since a clean-label attack's evaluation-eligible images are fewer than the whole pool, and the two halves are paired image for image.

In [2]:
loaders, manifest = build_psbd_loaders_from_checkpoint(CHECKPOINT, max_samples=MAX_SAMPLES)
architecture = detect_architecture(CHECKPOINT)
model = load_checkpoint(architecture, CHECKPOINT, device)
num_classes = DATASET_REGISTRY[manifest["dataset"]].num_classes

print("architecture:", architecture)
print("dataset:", manifest["dataset"], f"({num_classes} classes)")
print("probe attack:", manifest["probe_attack"], "target label", manifest["probe_target_label"])
for name, loader in loaders.items():
    print(f"  {name:11s} {len(loader.dataset):5d} images")

heldout = set(manifest["heldout_indices"])
analysis = set(manifest["analysis_clean_indices"])
print("validation and analysis disjoint:", heldout.isdisjoint(analysis))

/lustre/home/pstika/projects/PSBD-ViT/.venv/lib/python3.11/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


architecture: vit
dataset: cifar100 (100 classes)
probe attack: badnet_a2o target label 0
  validation    800 images
  clean         800 images
  backdoor      792 images
validation and analysis disjoint: True


## The unperturbed baseline

Every PSU is a drop measured against this pass, so it runs once, before any placement is attached.

In [3]:
baselines = {
    name: build_baseline_cache(model, loader, device, use_bfloat16=True)
    for name, loader in loaders.items()
}
baseline_probs = {n: torch.cat([b["probs"] for b in c]) for n, c in baselines.items()}
baseline_labels = {n: torch.cat([b["labels"] for b in c]) for n, c in baselines.items()}
loader_labels = {n: torch.cat([b["loader_labels"] for b in c]) for n, c in baselines.items()}

asr = (baseline_labels["backdoor"] == loader_labels["backdoor"]).float().mean().item()
print(f"attack success rate on the unperturbed model: {asr:.4f}")

attack success rate on the unperturbed model: 1.0000


## Attaching each placement and sweeping its rate ladder

`plug_dropout` attaches a fresh perturbation module at the named site through forward hooks and returns handles `unplug_dropout` removes it by. The model itself never switches to training mode, so its own dropout, if it has any, stays off.

In [4]:
def run_rate(position_names, operator, rate):
    handles = plug_dropout(
        model, architecture, position_names,
        {name: build_operator(operator) for name in position_names}, rate,
    )
    try:
        measured = {}
        for name, loader in loaders.items():
            probs, argmax = compute_dropout_pass_probs(
                model, loader, baseline_labels[name], device, FORWARD_PASSES, True, seed=0,
            )
            measured[name] = {
                "psu": psu_from_cache(baseline_probs[name], baseline_labels[name], probs),
                "psu_ratio": psu_ratio_from_cache(baseline_probs[name], baseline_labels[name], probs),
                "sigma": shift_ratio(baseline_labels[name], argmax),
                "argmax": argmax,
            }
    finally:
        unplug_dropout(handles)
    return measured


swept = {}
for placement_name, (position_names, operator) in PLACEMENTS.items():
    swept[placement_name] = {rate: run_rate(position_names, operator, rate) for rate in RATES}
    joined_positions = ", ".join(position_names)
    print(f"{placement_name} ({joined_positions}, {operator}) swept over {len(RATES)} rates")

PSBD-TM (before_attention_norm, token_mask) swept over 8 rates


PSBD-RD (after_attention_residual, after_mlp_residual, dropout) swept over 8 rates


## The rule and the threshold

The rate a placement is read at is the smallest one whose clean-validation predictions shift 80 percent of the time, `select_rate_adaptively`. The same rate ladder also gives an oracle rate, the best AUROC in hindsight, which a defender cannot legally choose, and a matched-shift rate used only to compare two placements at the same disturbance.

In [5]:
rule_rows = []
for placement_name in PLACEMENTS:
    shift_by_rate = {rate: m["validation"]["sigma"] for rate, m in swept[placement_name].items()}
    auroc_by_rate = {}
    for rate, measured in swept[placement_name].items():
        report = detection_report(
            measured["validation"]["psu_ratio"], measured["clean"]["psu_ratio"],
            measured["backdoor"]["psu_ratio"], HEADLINE_QUANTILE,
        )
        auroc_by_rate[rate] = report["auroc"]

    adaptive_rate = select_rate_adaptively(shift_by_rate)
    matched_rate = select_rate_at_matched_shift(shift_by_rate, PLACEMENT_MATCH_TARGET)
    oracle_rate = select_rate_by_oracle(auroc_by_rate)
    rule_rows.append(
        {
            "placement": placement_name,
            "adaptive_rate": adaptive_rate,
            "adaptive_auroc": auroc_by_rate.get(adaptive_rate),
            "matched_rate": matched_rate,
            "matched_auroc": auroc_by_rate.get(matched_rate),
            "oracle_rate": oracle_rate,
            "oracle_auroc": auroc_by_rate.get(oracle_rate),
        }
    )

rule_frame = pd.DataFrame(rule_rows).set_index("placement")
rule_frame.round(4)

,adaptive_rate,adaptive_auroc,matched_rate,matched_auroc,oracle_rate,oracle_auroc
placement,,,,,,
PSBD-TM,0.5,0.9894,0.4,0.9599,0.6,0.995
PSBD-RD,0.1,0.6120,0.1,0.6120,0.1,0.612


The oracle row is never worse than the deployable adaptive row, by construction, since it searches the same grid with the labels a real defender does not have. The gap between them is the cost of choosing a rate before seeing whether it worked.

## The PSU histograms

One histogram per placement, at the adaptive rate, on the clean and triggered halves of the analysis pool, with the threshold marked. This is the same figure `scripts/paper/fig_psu_histograms.py` builds from the cache on disk, for a handful of models chosen to show both a clear separation and a near-chance reading. Here it is drawn from the sweep just run, on one model, for both named placements side by side.

In [6]:
figure, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for axis, placement_name in zip(axes, PLACEMENTS):
    adaptive_rate = rule_frame.loc[placement_name, "adaptive_rate"]
    measured = swept[placement_name][adaptive_rate]
    threshold = threshold_at_quantile(measured["validation"]["psu_ratio"], HEADLINE_QUANTILE)
    auroc = rule_frame.loc[placement_name, "adaptive_auroc"]

    bin_edges = np.histogram_bin_edges(
        np.concatenate([measured["clean"]["psu_ratio"].numpy(), measured["backdoor"]["psu_ratio"].numpy()]),
        bins=30,
    )
    axis.hist(measured["clean"]["psu_ratio"].numpy(), bins=bin_edges, density=True, alpha=0.6, label="clean")
    axis.hist(measured["backdoor"]["psu_ratio"].numpy(), bins=bin_edges, density=True, alpha=0.6, label="triggered")
    axis.axvline(threshold, color="black", ls="--", lw=1.2)
    axis.set_title(f"{placement_name}, rate {adaptive_rate}, AUROC {auroc:.3f}")
    axis.set_xlabel("fractional PSU")

axes[0].set_ylabel("density")
axes[0].legend()
figure.tight_layout()
plt.show()

## Every quantile at once

AUROC is threshold free, so it is identical at every quantile. Only the true-positive and false-positive rate move, and the false-positive rate tracks the quantile almost exactly, which is the direct evidence that the threshold is doing exactly what it is asked to do.

In [7]:
quantile_rows = [
    {
        "quantile": q,
        **detection_report(
            swept["PSBD-TM"][rule_frame.loc["PSBD-TM", "adaptive_rate"]]["validation"]["psu_ratio"],
            swept["PSBD-TM"][rule_frame.loc["PSBD-TM", "adaptive_rate"]]["clean"]["psu_ratio"],
            swept["PSBD-TM"][rule_frame.loc["PSBD-TM", "adaptive_rate"]]["backdoor"]["psu_ratio"],
            q,
        ),
    }
    for q in PSBD_QUANTILES
]
pd.DataFrame(quantile_rows).set_index("quantile").round(4)

,threshold,tpr,fpr,auroc,auroc_two_sided,direction
quantile,,,,,,
0.01,0.2860,0.7247,0.0150,0.9894,0.9894,as_expected
0.05,0.7083,0.9987,0.0587,0.9894,0.9894,as_expected
0.10,0.8648,1.0000,0.1100,0.9894,0.9894,as_expected
0.15,0.9234,1.0000,0.1725,0.9894,0.9894,as_expected
0.20,0.9478,1.0000,0.2237,0.9894,0.9894,as_expected
0.25,0.9626,1.0000,0.2912,0.9894,0.9894,as_expected


## Where a shifted clean prediction lands

The original paper explains PSU by neuron bias: under perturbation the network falls back on the classes its neurons favour, and the backdoor's neurons favour the target class most firmly, so a shifted clean prediction should land on the target more often than chance. Notebook 04 tests this properly across the whole set of models. Here it is one histogram on this model alone.

In [8]:
adaptive_rate = rule_frame.loc["PSBD-TM", "adaptive_rate"]
histogram = shift_target_histogram(
    baseline_labels["clean"], swept["PSBD-TM"][adaptive_rate]["clean"]["argmax"], num_classes
)
counts = np.array(histogram)
target = manifest["probe_target_label"]

figure, axis = plt.subplots(figsize=(8, 3.6))
axis.bar(range(num_classes), counts, width=1.0)
axis.axvline(target, color="red", lw=1.5, label=f"attack target class {target}")
axis.set_xlabel("class a shifted clean prediction landed on")
axis.set_ylabel("count")
axis.legend()
figure.tight_layout()
plt.show()

share = counts[target] / counts.sum() if counts.sum() else float("nan")
print(f"shifted clean predictions landing on the target class: {share:.4f}")
print(f"uniform expectation:                                    {1 / num_classes:.4f}")

shifted clean predictions landing on the target class: 0.0000
uniform expectation:                                    0.0100
